# ISOM 835 · Session 5 — From Scores to Decisions
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Oct 19 · Prof. Hasan Arslan**

**Tonight in one sentence:** a model tells you *how likely* something is; **you** decide *what to do about it* — and that decision is where the money is.

**How to use this notebook:** run the cells top to bottom (Shift + Enter). Read the short text above each cell first: it tells you **what the cell does** and **how to read the output**. You do not need to understand every line of code tonight. You do need to understand every *result*.

### Six words you will hear tonight

| Word | Plain English |
|---|---|
| **Score** (probability) | The model's guess, between 0 and 1, that a customer will leave. 0.30 = "30% chance." |
| **Threshold** (cutoff) | The line we draw: everyone *above* it gets the offer. `predict()` uses 0.5 unless you say otherwise. |
| **Confusion matrix** | A 2 × 2 scorecard: who we flagged vs. who actually left. |
| **Profit curve** | Profit at every possible threshold. The highest point is the threshold to use. |
| **Calibration** | Honesty check: when the model says 30%, do about 30 out of 100 such customers really leave? |
| **Lift** | How many times better than picking customers at random. |

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## The business situation

You work at a phone and internet company (the Telco data from Session 1). Every month some customers **churn** (cancel).

- Marketing can send a **retention offer** that costs **$50** per customer.
- A customer who stays is worth about **$1,500** to the company.
- If someone who was about to leave gets the offer, **30%** of them stay.

So catching one customer who was about to leave is worth, on average, **30% × $1,500 = $450**. Sending the offer to someone who was going to stay anyway wastes **$50**.

**The question tonight:** *which* customers should get the offer?

The next cell loads the data and splits it: 80% to **train** the model, 20% held back to **test** it, like an exam the model has not seen.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, confusion_matrix, roc_auc_score, brier_score_loss
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV

# Load the customers
URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
df = pd.read_csv(URL)

# Inputs (X): seven columns about each customer. get_dummies turns text columns like Contract into 0/1 columns.
columns = ['tenure', 'MonthlyCharges', 'Contract', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']
X = pd.get_dummies(df[columns], drop_first=True)

# Target (y): 1 if the customer churned, 0 if they stayed
y = (df['Churn'] == 'Yes').astype(int)

# 80% to train, 20% to test
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=835)

# The two business numbers
OFFER = 50    # cost of sending one offer
SAVE = 450    # average value of catching one customer who was about to leave

print(f'{len(df):,} customers · {y.mean():.0%} churned · {len(X_te):,} customers in the test set')

## Quick recap: what logistic regression gives you

Forget odds and log-odds for tonight. Here is all you need:

1. You show the model a customer (tenure, contract type, monthly bill, …).
2. It gives back **one number between 0 and 1**: the chance that customer leaves.
3. `predict()` then turns that number into yes/no with a **0.5 cutoff**: above 0.5 means "will churn."

Step 3 is the problem. **Nobody at the company chose 0.5.** It is just the default.

The next cell trains the model and shows its score for five test customers next to what really happened.

In [ ]:
base = LogisticRegression(max_iter=2000)
base.fit(X_tr, y_tr)

# p = each test customer's churn probability
p = base.predict_proba(X_te)[:, 1]

sample = pd.DataFrame({
    'tenure (months)': X_te['tenure'].values[:5],
    'monthly bill': X_te['MonthlyCharges'].values[:5],
    'churn probability': p[:5].round(2),
    'gets offer at 0.5?': np.where(p[:5] >= 0.5, 'yes', 'no'),
    'actually churned?': np.where(y_te.values[:5] == 1, 'yes', 'no'),
})
sample

## 1. Who chose 0.5?

Once we pick a cutoff, every test customer lands in one of four boxes:

| | **We sent the offer** | **We did not** |
|---|---|---|
| **Customer was leaving** | ✅ **Caught**: worth +$450, minus the $50 offer | ❌ **Missed**: we lost them |
| **Customer was staying** | 💸 **Wasted offer**: −$50 | 👍 Correctly left alone: $0 |

So the profit of any cutoff is:

> **Profit = $450 × (customers caught) − $50 × (offers sent)**

The next cell builds that profit formula and scores the default 0.5 cutoff.

In [ ]:
def profit(y_true, y_pred):
    # Count the four boxes, then apply the business numbers
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    offers_sent = tp + fp
    return SAVE * tp - OFFER * offers_sent

flag = p >= 0.5                      # who gets the offer at the default cutoff
tn, fp, fn, tp = confusion_matrix(y_te, flag).ravel()
print('At the default cutoff of 0.5:')
print(f'  offers sent:        {tp + fp}')
print(f'  churners caught:    {tp}')
print(f'  churners missed:    {fn}')
print(f'  wasted offers:      {fp}')
print(f'  profit:             ${profit(y_te, flag):,}')

### When is one offer worth sending? Do it with dollars.

Take one customer with churn probability *p*:
- **Expected gain** from the offer = *p* × $450
- **Cost** of the offer = $50, whatever happens

Send the offer when the gain beats the cost: *p* × 450 > 50, which means **p > 50 / 450 ≈ 0.11**.

**In words:** a $50 offer pays off for any customer with more than about a **1-in-10** chance of leaving. That is far below 0.5. The next cell checks five customers. Watch where the answer flips from *skip* to *send*.

In [ ]:
for prob in [0.05, 0.10, 0.12, 0.30, 0.60]:
    gain = prob * SAVE
    decision = 'SEND' if gain > OFFER else 'skip'
    print(f'churn probability {prob:.2f}: expected gain ${gain:6.2f} vs. cost ${OFFER}  →  {decision}')

## 2. The profit curve

The rule above is for one customer at a time. Now we do it for the whole test set: **try every cutoff from 0.01 to 0.99**, compute the profit for each, and plot it.

**How to read the chart:** the left chart is profit, and its highest point is the best cutoff. The right chart is accuracy. They peak at **different** cutoffs, and the business gets paid in profit, not accuracy.

In [ ]:
cutoffs = np.linspace(0.01, 0.99, 99)
profits = []
accuracies = []
for t in cutoffs:
    flag = p >= t
    profits.append(profit(y_te, flag))
    accuracies.append((flag == y_te).mean())
profits = np.array(profits)
accuracies = np.array(accuracies)

best_t = cutoffs[profits.argmax()]
best_acc_t = cutoffs[accuracies.argmax()]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(cutoffs, profits, color='#2e8b57', lw=2)
ax[0].axvline(best_t, color='#f5a524', ls='--')
ax[0].set_title(f'Profit is highest at cutoff {best_t:.2f}')
ax[0].set_xlabel('cutoff'); ax[0].set_ylabel('profit ($)')
ax[1].plot(cutoffs, accuracies, color='#5b6bd8', lw=2)
ax[1].axvline(best_acc_t, color='#f5a524', ls='--')
ax[1].set_title(f'Accuracy is highest at cutoff {best_acc_t:.2f}')
ax[1].set_xlabel('cutoff'); ax[1].set_ylabel('accuracy')
plt.tight_layout(); plt.show()

profit_at_half = profits[np.abs(cutoffs - 0.5).argmin()]
print(f'profit at 0.50: ${profit_at_half:,}')
print(f'profit at {best_t:.2f}: ${profits.max():,}')
print(f'extra profit from moving the cutoff: ${profits.max() - profit_at_half:,} on {len(y_te):,} test customers')

## 3. Let scikit-learn find the cutoff: `TunedThresholdClassifierCV`

There is a catch in Step 2: we found the best cutoff **on the test set**. That is like studying with the exam answers, so the test score is no longer a fair grade.

`TunedThresholdClassifierCV` does the same search as Step 2, but **only on the training data** (using cross-validation from Session 3). The test set stays a fair exam. It takes three lines:

1. Turn our `profit` function into a scorer with `make_scorer`.
2. Wrap the model in `TunedThresholdClassifierCV` and tell it to maximize that scorer.
3. Fit, then read `best_threshold_`.

In [ ]:
profit_scorer = make_scorer(profit)
tuned = TunedThresholdClassifierCV(LogisticRegression(max_iter=2000), scoring=profit_scorer, cv=5)
tuned.fit(X_tr, y_tr)
print(f'cutoff chosen on the training data: {tuned.best_threshold_:.3f}\n')

print(f'profit with the 0.50 cutoff:   ${profit(y_te, base.predict(X_te)):>8,}')
print(f'profit with the tuned cutoff:  ${profit(y_te, tuned.predict(X_te)):>8,}')

# If the business gives you the cutoff instead, fix it with FixedThresholdClassifier
fixed = FixedThresholdClassifier(base, threshold=0.10).fit(X_tr, y_tr)
print(f'profit with a fixed 0.10:      ${profit(y_te, fixed.predict(X_te)):>8,}')

**Stop and notice:** the model did not change at all. We wrote down what each mistake costs, and moving the cutoff made the money.

## 4. Calibration: does "30%" really mean 30%?

Think of a weather forecast. If it says "80% chance of rain" on 100 days, it should rain on about 80 of them. If it rains on only 50, the forecast **ranks** days well but its **numbers** can't be trusted.

Our profit math multiplied the model's probabilities by dollars, so we need the numbers to be honest. The chart in the next cell checks three models:

- **How to read it:** each dot groups customers with similar scores. Across is what the model *said*; up is what *really happened*. **Dots on the dashed diagonal mean the model is honest.**
- **Brier score:** one number for the same idea, the average squared gap between the prediction and what happened. **Lower is better.**

In [ ]:
rf = RandomForestClassifier(300, min_samples_leaf=2, random_state=835, n_jobs=-1).fit(X_tr, y_tr)
gbm = HistGradientBoostingClassifier(random_state=835).fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(5.5, 5))
for name, model in [('logistic', base), ('random forest', rf), ('gradient boosting', gbm)]:
    CalibrationDisplay.from_estimator(model, X_te, y_te, n_bins=10, name=name, ax=ax)
    scores = model.predict_proba(X_te)[:, 1]
    print(f'{name:18s} AUC {roc_auc_score(y_te, scores):.3f}   Brier {brier_score_loss(y_te, scores):.4f}')
plt.show()

**If a model's dots bend away from the diagonal, fix it with `CalibratedClassifierCV`.** It learns a correction from the training data. The model's ranking barely changes, but its percentages get more honest. (Random forests and boosting need this more often than logistic regression.)

In [ ]:
rf_cal = CalibratedClassifierCV(RandomForestClassifier(300, min_samples_leaf=2, random_state=835, n_jobs=-1), method='isotonic', cv=5)
rf_cal.fit(X_tr, y_tr)

for name, model in [('random forest, raw', rf), ('random forest, calibrated', rf_cal)]:
    scores = model.predict_proba(X_te)[:, 1]
    print(f'{name:26s} AUC {roc_auc_score(y_te, scores):.3f}   Brier {brier_score_loss(y_te, scores):.4f}')

fig, ax = plt.subplots(figsize=(5.5, 5))
CalibrationDisplay.from_estimator(rf, X_te, y_te, n_bins=10, name='random forest, raw', ax=ax)
CalibrationDisplay.from_estimator(rf_cal, X_te, y_te, n_bins=10, name='random forest, calibrated', ax=ax)
plt.show()

## 5. Lift and gain: when the budget is fixed

Often marketing says: *"We can only afford to contact 10% of customers."* Then there is no cutoff to pick, only a question: **who are the best 10%?**

Answer: sort customers from highest churn score to lowest and contact from the top.

- **Gain chart (left):** if we contact the top X% of customers, what share of **all** churners do we reach? The dashed line is picking at random.
- **Lift (right):** how many times better than random each slice is. A lift of 2.5 means the model finds 2.5 × as many churners as random picking.

In [ ]:
order = np.argsort(-p)                          # customers from highest score to lowest
churned_sorted = y_te.values[order]
share_reached = np.cumsum(churned_sorted) / churned_sorted.sum()
share_contacted = np.arange(1, len(p) + 1) / len(p)

slices = np.arange(0.1, 1.01, 0.1)              # top 10%, top 20%, ..., 100%
reached = np.array([share_reached[int(len(p) * s) - 1] for s in slices])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(share_contacted, share_reached, color='#2ea89a', lw=2, label='model')
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8, label='random')
ax[0].set_xlabel('share of customers contacted'); ax[0].set_ylabel('share of churners reached'); ax[0].set_title('Gain chart'); ax[0].legend()
ax[1].bar(np.arange(10, 101, 10), reached / slices, width=7, color='#5b6bd8')
ax[1].set_xlabel('top X% contacted'); ax[1].set_ylabel('lift (× better than random)'); ax[1].set_title('Lift')
plt.tight_layout(); plt.show()

print(f'Contacting the top 10% reaches {reached[0]:.0%} of all churners: lift = {reached[0] / 0.1:.1f}')

The next cell answers the budget question directly: if we can contact only *k* customers, how many churners do we catch, and what is the profit?

In [ ]:
for k in [100, 200, 300, 500]:
    top_k = order[:k]
    caught = y_te.values[top_k].sum()
    print(f'contact top {k:3d}: {caught:3d} churners caught · profit ${SAVE * caught - OFFER * k:>7,} · hit rate {caught / k:.0%}')

## 6. The honest caveat: "likely to leave" ≠ "will be persuaded"

A churn model finds people **likely to leave**. But the offer only helps some of them. Picture four kinds of customers:

| Group | What happens if we send the offer |
|---|---|
| **Sure things**: staying anyway | $50 wasted |
| **Lost causes**: leaving no matter what | $50 wasted |
| **Persuadables**: stay *because of* the offer | ✅ the only group that pays |
| **Sleeping dogs**: the contact reminds them to cancel | ❌ we lose them |

Finding the persuadables is called **uplift modeling**. It needs an A/B test (offer vs. no offer) and is a topic for another course; you just need to know it exists. The next cell is a small what-if: suppose only 35% of churners can be persuaded.

In [ ]:
rng = np.random.default_rng(835)
persuadable = rng.random(len(p)) < 0.35          # pretend only 35% of churners respond to the offer
value = np.where(persuadable, SAVE, 0)           # catching a non-persuadable churner is worth $0

flag = tuned.predict(X_te).astype(bool)
caught_value = (value * ((y_te.values == 1) & flag)).sum()
print(f'profit we expected at the tuned cutoff:     ${profit(y_te, flag):,}')
print(f'profit if only persuadables respond:        ${caught_value - OFFER * flag.sum():,}')

## 7. Midterm Model Competition: launch

Teams of 2–3. You get hotel bookings; predict which ones will be **canceled** (`is_canceled`). Kaggle scores your predictions on a hidden test set: half shows on the public leaderboard now, and the other half (the private leaderboard) decides the final ranking.

- **Leaderboard closes Mon Nov 9, 4:30 PM.** Upload the same predictions file on Canvas by then.
- **Memo due Mon Nov 9, 11:59 PM:** one page on what you tried, what worked, and what leaked.
- **Grade:** 50% private-leaderboard band, 50% memo.
- **Invitation link:** on Canvas. Start from the competition starter notebook; it makes a valid submission in a few minutes.

## 8. Your turn

1. **Cost sensitivity.** Marketing raises the offer to **$200**. In the setup cell change `OFFER = 50` to `OFFER = 200`, then re-run all cells (Runtime → Run all).
   - What is the new tuned cutoff? Did it go **up or down**?
   - Explain why in one sentence. *Hint: redo the dollar check from Step 1: when is p × $450 > $200?*
2. **Budget memo.** Marketing can afford **250** contacts. Using the Step 5 idea, how many churners do you catch and what is the profit? Write two or three sentences to the marketing manager, in plain English, with no code words.
3. **Challenge (optional).** Fit `HistGradientBoostingClassifier(class_weight='balanced', random_state=835)`. Compare its AUC and Brier score before and after `CalibratedClassifierCV`. Did the percentages get more honest? Why might "balanced" weights hurt calibration?

In [ ]:
# Your turn — work here

## What we learned tonight

- **A model gives a score; the business makes the decision.** With a $50 offer and $450 per customer saved, the offer pays whenever p > 50 / 450 ≈ 0.11, not 0.5.
- **Pick the cutoff with dollars, not accuracy.** The profit curve shows the best one; `TunedThresholdClassifierCV` finds it without peeking at the test set.
- **Check that the percentages are honest** (calibration) before you multiply them by dollars.
- **With a fixed budget, contact from the top of the list**: gain and lift show how much better than random that is.
- **Likely to leave ≠ persuadable.**

**HW3** (due Mon Nov 16, 11:59 PM): Part 1 is tonight's ideas on your HW2 credit model; Part 2, Ames regression, arrives next week. **Competition:** team formed and one submission by Mon Nov 2. **Project Milestone 1** (your problem and dataset, one paragraph): Mon Nov 2.